# Flip set (stage A)

Items for the mechanistic comparison, built from the corrected (rescored) evals at the phase
checkpoints in `checkpoints.yaml`. Only items that pass the Layer-0 filter at **every** phase
checkpoint of a run are eligible, so every comparison is on the same items.

Groups per run:
- **flip** -- CTX at peak, PAR at trough (the phenomenon). For tulu, split by what happens at
  recovery: `flip_recovered` (CTX again) vs `flip_stuck` (still PAR).
- **stable_ctx** -- CTX at every phase checkpoint. Controls: capitals in this group show it
  isn't just "being a capitals item"; other strata show general context-following.

Runs locally (no GPU). Writes `flip_set.json`, which the Colab notebooks read.

In [ ]:
import json, yaml
from collections import Counter

cfg = yaml.safe_load(open("checkpoints.yaml"))

def phase_steps(run):
    return {k: int(v.split("-")[1]) for k, v in cfg["runs"][run]["phases"].items() if v}

def load_eval(run, step):
    f = f"../../results/cpi-results/{run}-results/checkpoint_step{step}_metrics.json"
    d = json.load(open(f, encoding="utf-8"))
    assert d.get("rescored") is True, f"{f} is not rescored -- labels/deltas may carry the tokenisation bug"
    return {r["item_id"]: r for r in d["per_item"]}

flip_set = {}
for run in ["tulu", "alpaca"]:
    phases = phase_steps(run)
    evals = {name: load_eval(run, step) for name, step in phases.items()}
    ids = sorted(set.intersection(*[{i for i, r in e.items() if r["passed"]} for e in evals.values()]))

    groups = {}
    for i in ids:
        lab = {name: evals[name][i]["final_label"] for name in phases}
        if lab["peak"] == "CTX" and lab["trough"] == "PAR":
            g = "flip"
            if "recovery" in lab:
                g = "flip_recovered" if lab["recovery"] == "CTX" else "flip_stuck"
        elif all(v == "CTX" for v in lab.values()):
            g = "stable_ctx"
        else:
            continue
        groups.setdefault(g, []).append({"item_id": i, "stratum": i.split("_")[0], "labels": lab})

    flip_set[run] = {"phases": phases, "n_eligible": len(ids), "groups": groups}

json.dump(flip_set, open("flip_set.json", "w"), indent=1)

for run, fs in flip_set.items():
    print(f"=== {run}  phases={fs['phases']}  eligible={fs['n_eligible']}")
    for g, rows in fs["groups"].items():
        print(f"  {g:15s} n={len(rows):3d}  by stratum: {dict(Counter(r['stratum'] for r in rows))}")

## Validation

1. **Label basis** -- was each label set by the logprob score or by the judge, and does the
   generated text (the judge-free `ordered` label) agree?
2. **Stability** -- a label seen at one checkpoint could be noise. Each item is marked `robust`
   if it holds its group's label at most checkpoints of every rescored phase window.
3. **Controls** -- robust flipping capitals vs robust stable capitals should match on answer
   length and no-context strength, so any mechanistic difference isn't one of those.
4. **Eyeball** -- read what the model actually generated at peak and trough.

In [ ]:
import random, statistics

# rescored checkpoints around each phase pick (only these have trustworthy per-item labels)
WINDOWS = {"tulu":   {"peak": [400, 450, 500, 550, 600], "trough": [3050, 3100, 3150, 3200, 3250],
                      "recovery": [4900, 4950, 5000]},
           "alpaca": {"peak": [50, 100, 150, 200], "trough": [700, 750, 800]}}
base = {r["item_id"]: r for r in json.load(open("parametric-strength/base.json", encoding="utf-8"))["per_item"]}

def expected(group, phase):
    if phase == "peak":
        return "CTX"
    if phase == "trough":
        return "PAR" if group.startswith("flip") else "CTX"
    return "PAR" if group == "flip_stuck" else "CTX"          # recovery

random.seed(0)
for run, fs in flip_set.items():
    ev = {s: load_eval(run, s) for steps in WINDOWS[run].values() for s in steps}
    ph = fs["phases"]
    print(f"\n==================== {run} ====================")

    for g, rows in fs["groups"].items():
        for r in rows:        # 2. stability -> robust flag
            r["robust"] = all(sum(ev[s][r["item_id"]]["final_label"] == expected(g, p) for s in steps) > len(steps) / 2
                              for p, steps in WINDOWS[run].items())
        n_rob = sum(r["robust"] for r in rows)
        print(f"{g:15s} robust {n_rob:3d}/{len(rows):3d}  (capitals {sum(r['robust'] and r['stratum'] == 'cap' for r in rows)})")

    flips = [r["item_id"] for g, rows in fs["groups"].items() if g.startswith("flip") for r in rows]
    for p in ["peak", "trough"]:        # 1. label basis
        e = ev[ph[p]]
        by_judge = sum(e[i]["logprob"]["label"] == "AMBIG" for i in flips)
        gen_agree = sum(e[i]["ordered"]["label"] == e[i]["final_label"] for i in flips)
        print(f"flips at {p:6s}: judge decided {by_judge}/{len(flips)}, generation agrees {gen_agree}/{len(flips)}")

    rob_flip = [r["item_id"] for g, rows in fs["groups"].items() if g.startswith("flip")
                for r in rows if r["robust"] and r["stratum"] == "cap"]
    rob_stab = [r["item_id"] for r in fs["groups"]["stable_ctx"] if r["robust"] and r["stratum"] == "cap"]
    for k in ["n_par", "par_lp_pt"]:    # 3. controls
        print(f"capitals {k:9s}: robust flip median {statistics.median(base[i][k] for i in rob_flip):.2f}"
              f"  vs stable median {statistics.median(base[i][k] for i in rob_stab):.2f}")

    for i in random.sample(rob_flip, 3):  # 4. eyeball
        a, b = ev[ph["peak"]][i], ev[ph["trough"]][i]
        print(f"  {i}: {a['question']}  par={a['parametric_answer']}  cf={a['counterfactual_answer']}")
        print(f"     peak   -> {a['response'][:60]!r}")
        print(f"     trough -> {b['response'][:60]!r}")

json.dump(flip_set, open("flip_set.json", "w"), indent=1)
print("\nflip_set.json updated with per-item 'robust' flags")